# JW $Z_4^{TF}$ system - cocycle extraction

Created: 07-09-2026

Objectives:
* Iterate on [this notebook](jw_z_4_tf_system_improved_disentanglers.ipynb), now extract $\nu_2, \omega_2$ (depending on your convention) and extract associated invariants, coboundaries, etc.

# Imports

In [1]:
import numpy as np

In [2]:
import jax
jax.config.update('jax_platform_name', 'cpu')

import jax.numpy as jnp

In [3]:
import matplotlib.pyplot as plt

In [4]:
from tqdm import tqdm

In [5]:
from functools import reduce
from operator import mul
from itertools import product

In [6]:
from random import random

In [7]:
import quimb.tensor as qtn
import quimb as qu

In [8]:
from scipy.stats import unitary_group

In [9]:
from collections import Counter

In [10]:
import pandas as pd

In [11]:
from time import time

In [12]:
from humanize import naturalsize

# Definitions
## Construct cluster state

In [13]:
np_up_X_state = 1/(np.sqrt(2))*np.array([1,1])

In [14]:
qu_up_X_state = qtn.Tensor(
    data=np_up_X_state,
    inds=('k',),
    tags='prod'
)

In [15]:
np_CZ = np.diag([1,1,1,-1])

In [16]:
np_CZ = np_CZ.reshape((2,)*4)

In [17]:
qu_CZ = qtn.Tensor(
    data=np_CZ,
    inds=('k1', 'k2', 'b1', 'b2'),
    tags='CZ'
)

In [18]:
np_hadamard = np.pow(2, -1/2)*np.array([
    [1,1],
    [1,-1]
])

In [19]:
qu_hadamard = qtn.Tensor(data=np_hadamard, inds=('k', 'b'), tags='Had')

In [20]:
def get_cluster_state_qu_tensor_network(num_sites):
    assert (num_sites%2) == 0

    product_state_tensors = [
        qu_up_X_state.reindex({'k': f'kc_1_{i}'})
        for i in range(num_sites)
    ]

    first_layer_circuit_tensors = [
        qu_CZ.reindex({
            'b1': f'kc_1_{i}',
            'b2': f'kc_1_{i+1}',
            'k1': f'kc_2_{i}',
            'k2': f'kc_2_{i+1}'
        })
        for i in range(0, num_sites, 2)
    ]


    second_layer_circuit_tensors = [
        qu_CZ.reindex({
            'b1': f'kc_2_{i}',
            'b2': f'kc_2_{(i+1)%num_sites}',
            'k1': f'kh_{i}',
            'k2': f'k{(i+1)%num_sites}'
        })
        for i in range(1, num_sites+1, 2)
    ]

    hadamard_layer = [
        qu_hadamard.reindex({
            'b': f'kh_{i}',
            'k': f'k{i}'
        })
        for i in range(1, num_sites, 2)
    ]
    all_tensors = (
        product_state_tensors
        + first_layer_circuit_tensors
        + second_layer_circuit_tensors
        + hadamard_layer
    )

    out = qtn.TensorNetwork(all_tensors, virtual=True)
    out.mangle_inner_()

    return out

## Construct product state

In [21]:
np_up_X_state = 1/(np.sqrt(2))*np.array([1,1])

In [22]:
np_up_Z_state = np.array([1,0])

In [23]:
qu_up_Z_state = qtn.Tensor(
    data=np_up_Z_state,
    inds=('k',),
    tags='prod'
)

In [24]:
alternating_states = [
    qu_up_X_state,
    qu_up_Z_state
]

def get_product_qu_tensor_network(num_sites):
    assert (num_sites%2) == 0

    product_state_tensors = [
        alternating_states[i%2].reindex({'k': f'k{i}'})
        for i in range(num_sites)
    ]

    out = qtn.TensorNetwork(
        product_state_tensors,
        virtual=True
    )
    out.mangle_inner_()

    return out

## Symmetries

In [25]:
def multikron(arrays):
    return reduce(np.kron, arrays)

In [26]:
np_I = np.array([
    [1,0],
    [0,1]
])

np_X = np.array([
    [0,1],
    [1,0]
])

np_Y = np.array([
    [0,-1j],
    [1j,0]
])

np_Z = np.array([
    [1,0],
    [0,-1]
])

In [27]:
qu_I = qtn.Tensor(
    np_I,
    inds=['k', 'b'],
    tags='X'
)

qu_X = qtn.Tensor(
    np_X,
    inds=['k', 'b'],
    tags='X'
)

qu_Y = qtn.Tensor(
    np_Y,
    inds=['k', 'b'],
    tags='Y'
)

qu_Z = qtn.Tensor(
    np_Z,
    inds=['k', 'b'],
    tags='Z'
)

In [28]:
def get_multisite_qu_X(num_sites):
    np_many_X = multikron([np_X]*num_sites)

    out = qtn.Tensor(
        np_many_X,
        inds=['k', 'b'],
        tags='mulit_site_X',
    )

    return out

In [29]:
def get_multisite_qu_I(num_sites):
    np_many_I = multikron([np_I]*num_sites)

    out = qtn.Tensor(
        np_many_I,
        inds=['k', 'b'],
        tags='mulit_site_I',
    )

    return out

In [30]:
qu_spin_fermion_fp = (
    qu_I.reindex({'k': 'ks', 'b': 'bs'})
    & qu_Z.reindex({'k': 'kf', 'b': 'bf'})
).contract()

qu_unit_cell_fp = qu_spin_fermion_fp.fuse({
    'k': ['ks', 'kf'],
    'b': ['bs', 'bf']
})

In [31]:
"""
def get_multisite_qu_fp(num_sites):
    assert (num_sites%2)==0

    num_unit_cells = (num_sites//2)

    np
"""

'\ndef get_multisite_qu_fp(num_sites):\n    assert (num_sites%2)==0\n\n    num_unit_cells = (num_sites//2)\n\n    np\n'

Decompose T symmetry as $MK$:

In [32]:
np_00 = np.array([[1,0], [0,0]])
np_11 = np.array([[0,0], [0,1]])

In [33]:
def tensor_product_operators(op_1, op_2):
    out = (
        op_1[..., np.newaxis, np.newaxis]
        *op_2[np.newaxis, np.newaxis, ...]
    )

    return out

In [34]:
np_M = (
    tensor_product_operators(np_X, np_00)
    + tensor_product_operators(np_Y, np_11)
)

In [35]:
qu_M = qtn.Tensor(
    np_M,
    inds=['ks', 'bs', 'kf', 'bf']
)

In [36]:
np_M_reindexed = (
    qu_M
    .fuse({
        'k': ['ks', 'kf'],
        'b': ['bs', 'bf']
    })
    .transpose('k', 'b')
    .data
)

## Extracting projectors

In [37]:
def random_uniform_complex(shape):
    return np.random.uniform(size=shape) + 1j*np.random.uniform(size=shape)

In [38]:
def maximize_projector_states(rho, left_sites, proj_sites, right_sites):
    v = qtn.Tensor(
        data=random_uniform_complex((2,)*(len(proj_sites)-1)),
        inds=[f'k{i}' for i in proj_sites[:-1]]
    )

    tnopt = qtn.TNOptimizer(
        v,  # the tensor network we want to optimize
        loss_func,  # the function we want to minimize
        norm_fn=normalize_v,
        loss_constants={"rho": rho},
        loss_kwargs={
            "left_sites": left_sites,
            "proj_sites": proj_sites,
            "right_sites": right_sites,
        },
        autodiff_backend="jax",
        optimizer="L-BFGS-B",
        progbar=False
    )

    v_opt = tnopt.optimize(n=2000)

    embedded_v_opt = embed_fp_even_vector(v_opt).contract()

    return v_opt, embedded_v_opt, tnopt.losses

In [39]:
def projector_state_check(state, rho, left_sites):
    left_right_rho = (
        rho
        & state
        & state.conj().reindex({s: f'b{s[1:]}' for s in state.inds})
    )

    left_right_rho = left_right_rho.contract()

    left_inds = [
        f'{s}{i}'
        for i in left_sites
        for s in 'kb'
    ]

    schmidt_decomp = qtn.tensor_core.tensor_split(
        left_right_rho,
        left_inds=left_inds,
        method='svd',
        #cutoff=1e-6,
        cutoff_mode='abs',
        absorb=None,
        renorm=False,
        bond_ind='v'
    )

    schmidt_vals = schmidt_decomp.tensors[1]

    return schmidt_vals

### Embed vector

In [40]:
np_CX = (
    tensor_product_operators(np_00, np_I)
    + tensor_product_operators(np_11, np_X)
)

In [41]:
qu_CX = qtn.Tensor(
    np_CX,
    inds=['k1', 'b1', 'k2', 'b2']
)

In [42]:
np_up_Z_state = np.array([1,0])

In [43]:
qu_up_Z_state = qtn.Tensor(
    data=np_up_Z_state,
    inds=('k',),
    tags='Z0_pad'
)

In [44]:
def embed_fp_even_vector(v):
    # Take a vector of length 2N-1, and return a vector of length 2N
    # which commutes with IZIZ...IZIZ
    # Assuming v site ordering is spin-fermion-spin-...-fermion.

    sites = sorted(int(s[1:]) for s in v.inds)
    padded_site = sites[-1] + 1
    num_cx_gates = len(sites)//2
    
    if num_cx_gates>=1:
        padded_v = (
            v
            & qu_up_Z_state.reindex({'k': f'k{padded_site}_0'})
        )
    else:
        padded_v = (
            v
            & qu_up_Z_state.reindex({'k': f'k{padded_site}'})
        )
    num_cx_gates = len(sites)//2
    
    cx_gates = [
        qu_CX.reindex({
            'k1': f'k{sites[2*i+1]}',
            'b1': f'b{sites[2*i+1]}',
            'k2': f'k{padded_site}_{i+1}',
            'b2': f'k{padded_site}_{i}'
        })
        for i in range(num_cx_gates-1)
    ]
    
    # Handling annoying edge case logic
    if num_cx_gates >= 1:
        i = num_cx_gates-1
        cx_gates.append(
            qu_CX.reindex({
                'k1': f'k{sites[2*i+1]}',
                'b1': f'b{sites[2*i+1]}',
                'k2': f'k{padded_site}',
                'b2': f'k{padded_site}_{i}'
            })
        )
    
    reindexed_padded_v = (
        padded_v
        .reindex({
            f'k{sites[2*i+1]}': f'b{sites[2*i+1]}'
            for i in range(num_cx_gates)
        })
    )
    sym_v = qtn.TensorNetwork([
        reindexed_padded_v,
        *cx_gates
    ])
    
    sym_v.mangle_inner_()

    return sym_v

### Loss function

In [45]:
def get_rho_purity(rho, sites):
    # Assuming rho is a Hermitian reduced density matrix
    reindex_map = (
        {f'k{i}': f'b{i}' for i in sites}
        | {f'b{i}': f'k{i}' for i in sites}
    )

    rho_other = rho.reindex(reindex_map)
    #rho_other.mangle_inner_()
    
    out = (rho & rho_other).contract()

    return out

In [46]:
def normalize_v(v):
    norm = (v & v.conj()).contract()
    w = v*jnp.power(norm, -0.5)
    return w

In [47]:
def loss_func(v, rho, left_sites, proj_sites, right_sites):
    embed_v = embed_fp_even_vector(v)
    
    rho_lr = (
        rho
        & embed_v.reindex({f'k{i}': f'b{i}' for i in proj_sites})
        & embed_v.conj()
    )
    rho_lr = rho_lr.contract()
    tr_rho_lr = (
        rho_lr
        .reindex({f'k{i}': f'b{i}' for i in left_sites + right_sites})
        .contract()
    )
    
    rho_l = rho_lr.reindex(
        {f'k{i}': f'b{i}' for i in right_sites}
    )
    rho_l = rho_l.contract()*jnp.power(tr_rho_lr, -0.5)
    
    rho_r = rho_lr.reindex(
        {f'k{i}': f'b{i}' for i in left_sites}
    )
    rho_r = rho_r.contract()*jnp.power(tr_rho_lr, -0.5)
    
    purity_lr = get_rho_purity(rho_lr, left_sites + right_sites)
    purity_l = get_rho_purity(rho_l, left_sites)
    purity_r = get_rho_purity(rho_r, right_sites)
    
    reindex_map = (
        {f'k{i}': f'b{i}' for i in left_sites+right_sites}
        | {f'b{i}': f'k{i}' for i in left_sites+right_sites}
    )
    
    cross_term = (
        rho_lr.reindex(reindex_map)
        & rho_l
        & rho_r
    )
    cross_term = cross_term.contract()

    out = jnp.real(
        (purity_lr + purity_l*purity_r - 2*cross_term)/
        purity_lr
    )

    return out

## Extract cut rho and EDM

In [48]:
def generate_edm_from_cut_state(cut_state, sites, num_defect_sites):
    # Lots of duplicate code, probably a better way to do this.
    assert 2*num_defect_sites < len(sites)
    assert (num_defect_sites%2) == 0
    assert (len(sites)%2) == 0
    assert (sites[0]%2) == 0

    left_defect_sites = sites[:num_defect_sites]
    right_defect_sites = sites[-num_defect_sites:]
    internal_sites = sites[num_defect_sites:-num_defect_sites]

    # Being sloppy with the gate indices as the symmetries are invariant
    # under transpose and conjugation.
    left_sym_gates = [
        qu_M.reindex({
            'ks': f'b{i}',
            'kf': f'b{i+1}',
            'bs': f'c{i}',
            'bf': f'c{i+1}'
        })
        for i in left_defect_sites[::2]
    ]

    inner_gates = [
        qu_M.reindex({
            'ks': f'k{i}',
            'kf': f'k{i+1}',
            'bs': f'b{i}',
            'bf': f'b{i+1}'
        })
        for i in internal_sites[::2]
    ]

    right_sym_gates = [
        qu_M.reindex({
            'ks': f'b{i}',
            'kf': f'b{i+1}',
            'bs': f'c{i}',
            'bf': f'c{i+1}'
        })
        for i in right_defect_sites[::2]
    ]

    reindex_map = (
        {
            f'k{i}': f'c{i}'
            for i in (left_defect_sites + right_defect_sites)
        }
        |
        {
            f'k{i}': f'b{i}'
            for i in internal_sites
        }
    )

    # The fact that we don't conjugate the reindexed cut_state means that
    # we are effectively implementing local complex conjugation.
    edm = (
        cut_state
        & cut_state.reindex(reindex_map)
        & left_sym_gates
        & inner_gates
        & right_sym_gates
    )

    edm = edm.contract()

    fuse_maps = [
        ('k_left', (f'k{i}' for i in left_defect_sites)),
        ('b_left', (f'b{i}' for i in left_defect_sites)),
        ('k_right', (f'k{i}' for i in right_defect_sites)),
        ('b_right', (f'b{i}' for i in right_defect_sites))
    ]

    edm.fuse(fuse_maps, inplace=True)

    return edm

## Defect operators

In [49]:
def random_uniform_complex(shape):
    return np.random.uniform(size=shape) + 1j*np.random.uniform(size=shape)

In [50]:
def solve_for_boundary_operators(edm, num_iters=100):
    # Careful, the indices are reversed here for ease.
    # i.e. the k, b indices have been swapped to make tensor contraction easier.
    scores = list()

    u_left = qtn.tensor_builder.rand_tensor(
        (edm.ind_size('b_left'), edm.ind_size('k_left')),
        inds=['k_left', 'b_left'],
        dtype='complex64'
    )

    u_right = qtn.tensor_builder.rand_tensor(
        (edm.ind_size('b_right'), edm.ind_size('k_right')),
        inds=['k_right', 'b_right'],
        dtype='complex64'
    )

    for _ in range(num_iters):
        right_edm = (edm & u_left).contract()
        data = right_edm.data
        U, S, VH = np.linalg.svd(data)
        scores.append(np.sum(S))
    
        sol = (U @ VH).conj().T
        u_right = qtn.Tensor(sol, inds = ['b_right', 'k_right'])

        left_edm = (edm & u_right).contract()
        data = left_edm.data
        U, S, VH = np.linalg.svd(data)
        scores.append(np.sum(S))
    
        sol = (U @ VH).conj().T
        u_left = qtn.Tensor(sol, inds = ['b_left', 'k_left'])

    return (u_left, u_right), scores

## Apply random unitary to groundstate

In [51]:
def generate_random_su2():
    # Randomly sample a unitary, and scale by the determinant.
    u = unitary_group.rvs(2)
    det_u = np.linalg.det(u)
    su = u*np.power(det_u, -0.5)

    return su

In [52]:
X =  generate_random_su2()

In [53]:
np.linalg.det(X)

np.complex128(1.0000000000000002+1.9428902930940244e-16j)

In [54]:
np.round(X @ (X.conj().T), 3)

array([[ 1.+0.j, -0.+0.j],
       [-0.-0.j,  1.+0.j]])

In [55]:
def generate_random_symmetry_respecting_unitary_no_offset():
    # Generate a unitary which commutes with MK, where
    # M = X tensor (|0><0|) + Y tensor (|1><1|)

    phi = np.random.uniform(0, 2*np.pi)
    phi_phasor = np.exp(1j*phi)
    u0 = np.diag([phi_phasor, phi_phasor.conj()])

    if random() > 0.5:
        u0 = u0 @ np_X

    u1 = generate_random_su2()

    u = (
        tensor_product_operators(u0, np_00)
        + tensor_product_operators(u1, np_11)
    )

    qu_u = qtn.Tensor(
        u,
        inds=['ks', 'bs', 'kf', 'bf']
    )
    
    return qu_u

In [56]:
def generate_random_symmetry_respecting_unitary_offset():
    # Generate a unitary U such that IUI commutes with (MM)K, where
    # M = X tensor (|0><0|) + Y tensor (|1><1|), and concatenation denotes
    # tensor product.
    phi = np.random.uniform(0, 2*np.pi)
    phi_phasor = np.exp(1j*phi)
    u0 = np.diag([phi_phasor, phi_phasor.conj()])

    phi = np.random.uniform(0, 2*np.pi)
    phi_phasor = np.exp(1j*phi)
    u1 = np.diag([phi_phasor, phi_phasor.conj()])

    u = (
        tensor_product_operators(np_00, u0)
        + tensor_product_operators(np_11, u1)
    )

    qu_u = qtn.Tensor(
        u,
        inds=['kf', 'bf', 'ks', 'bs']
    )
    
    return qu_u

In [57]:
def generate_random_symmetry_respecting_unitary(offset):
    if offset:
        return generate_random_symmetry_respecting_unitary_offset()
    else:
        return generate_random_symmetry_respecting_unitary_no_offset()

In [58]:
# Warning, likely making assupmtions about shape of psi, number of sites being even here etc.
def apply_haar_random_fdlu_to_quimb_state(psi, domains_dict):
    num_sites = domains_dict['num_system_sites']

    depth = domains_dict['fdlu_depth']
    offset = domains_dict['fdlu_offset']
    all_circuit_lists = [
        list() for _ in range(depth)
    ]

    for layer, circuit_list in enumerate(all_circuit_lists):
        delta = layer
        is_offset = ((offset + delta)%2 == 1)

        for i in range(num_sites//2):
            site_1 = ((2*i)+delta+offset)%num_sites
            site_2 = ((2*i)+1+delta+offset)%num_sites

            u = generate_random_symmetry_respecting_unitary(is_offset)

            if is_offset:
                reindex_map = {
                    'kf': f'k_{layer+1}_{site_1}',
                    'ks': f'k_{layer+1}_{site_2}',
                    'bf': f'k_{layer}_{site_1}',
                    'bs': f'k_{layer}_{site_2}'
                }
            else:
                reindex_map = {
                    'ks': f'k_{layer+1}_{site_1}',
                    'kf': f'k_{layer+1}_{site_2}',
                    'bs': f'k_{layer}_{site_1}',
                    'bf': f'k_{layer}_{site_2}'
                }
            
            qu_u = u.reindex(reindex_map)

            circuit_list.append(qu_u)

    all_tensors = (
        [psi.reindex({f'k{i}': f'k_0_{i}' for i in range(num_sites)})]
        + sum(all_circuit_lists, start=[])
    )

    out = (
        qtn
        .TensorNetwork(all_tensors, virtual=False)
        .mangle_inner_()
        .reindex({f'k_{depth}_{i}': f'k{i}' for i in range(num_sites)}) 
    )

    return out

In [59]:
def extract_time_reversal_information_after_random_fdlu(psi, domains_dict,
    num_random_states=20):

    out = list()

    for _ in range(num_random_states):
        rand_psi = apply_haar_random_fdlu_to_quimb_state(psi, domains_dict)
        out.append(extract_time_reversal_information(rand_psi, domains_dict))

    return out

In [60]:
def extract_factorization_time_reversal_information_after_random_fdlu(psi,
    domains_dict, num_random_states=20):
    out = list()

    for _ in range(num_random_states):
        rand_psi = apply_haar_random_fdlu_to_quimb_state(psi, domains_dict)
        data = extract_factorization_time_reversal_information(
            rand_psi,
            domains_dict
        )
        out.append(data)

    return out

In [61]:
def get_quimb_psi_from_quspin_psi(quspin_psi):
    quimb_psi = qtn.Tensor(
        quspin_psi[::-1].reshape((2,)*num_sites),
        inds=[f'k{i}' for i in range(num_sites)]
    )

    return quimb_psi

## Sweep function

In [62]:
def extract_projector(psi, rho_sites, num_pad_sites, jw_even=False):
    proj_sites = rho_sites
    left_sites = list(range(
        min(rho_sites) - num_pad_sites,
        min(rho_sites)
    ))
    right_sites = list(range(
        max(rho_sites)+1,
        max(rho_sites)+num_pad_sites+1
    ))
    all_sites = left_sites + proj_sites + right_sites
    rho = (psi & psi.conj().reindex({f'k{i}': f'b{i}' for i in all_sites}))
    raw_proj_vec, proj_vec, losses = maximize_projector_states(
        rho,
        left_sites,
        proj_sites,
        right_sites
    )

    schmidt_vals = projector_state_check(
        proj_vec,
        rho,
        left_sites
    )

    return raw_proj_vec, proj_vec, losses, schmidt_vals

In [249]:
def find_invariants_via_projectors_from_random_state(psi, domains_dict, jw_even=False):
    rand_psi = apply_haar_random_fdlu_to_quimb_state(psi, domains_dict)
    
    raw_left_proj_vec, left_proj_vec, *left_proj_vec_results = extract_projector(
        rand_psi,
        domains_dict['left_projector_sites'],
        domains_dict['num_projector_pad_sites'],
        jw_even
    )
    
    raw_right_proj_vec, right_proj_vec, *right_proj_vec_results = extract_projector(
        rand_psi,
        domains_dict['right_projector_sites'],
        domains_dict['num_projector_pad_sites'],
        jw_even
    )
    
    cut_sites = list(range(
        min(domains_dict['left_projector_sites']),
        max(domains_dict['right_projector_sites'])+1
    ))
    
    cut_rho_unprojected = (
        rand_psi
        & rand_psi.conj().reindex({f'k{i}': f'b{i}' for i in cut_sites})
    )
    
    left_proj_sites = domains_dict['left_projector_sites']
    right_proj_sites = domains_dict['right_projector_sites']
    
    cut_rho = (
        cut_rho_unprojected.reindex({
            **{f'k{i}': f'l{i}' for i in left_proj_sites},
            **{f'k{i}': f'l{i}' for i in right_proj_sites},
            **{f'b{i}': f'c{i}' for i in left_proj_sites},
            **{f'b{i}': f'c{i}' for i in right_proj_sites}
        })
        & left_proj_vec.conj().reindex({f'k{i}': f'l{i}' for i in left_proj_sites})
        & left_proj_vec
        & left_proj_vec.reindex({f'k{i}': f'c{i}' for i in left_proj_sites})
        & left_proj_vec.conj().reindex({f'k{i}': f'b{i}' for i in left_proj_sites})
        & right_proj_vec.conj().reindex({f'k{i}': f'l{i}' for i in right_proj_sites})
        & right_proj_vec
        & right_proj_vec.reindex({f'k{i}': f'c{i}' for i in right_proj_sites})
        & right_proj_vec.conj().reindex({f'k{i}': f'b{i}' for i in right_proj_sites})
    )
    
    cut_rho_trace = (
        cut_rho
        .reindex({f'b{i}': f'k{i}' for i in cut_sites})
        .contract()
    )
    
    cut_rho = cut_rho/cut_rho_trace
    
    tranpose_map = (
        {f'k{i}': f'b{i}' for i in cut_sites}
        | {f'b{i}': f'k{i}' for i in cut_sites}
    )
    
    cut_rho_purity = (
        (cut_rho & cut_rho.reindex(tranpose_map))
        .contract()
    )
    
    sub_cut_sites = list(range(
        max(domains_dict['left_projector_sites'])+1,
        min(domains_dict['right_projector_sites'])
    ))
    
    sub_cut_rho = (
        cut_rho
        .reindex({f'k{i}': f'b{i}' for i in left_proj_sites + right_proj_sites})
        .contract()
    )
    
    sub_cut_rho_trace = (
        sub_cut_rho
        .reindex({f'b{i}': f'k{i}' for i in sub_cut_sites})
        .contract()
    )
    
    tranpose_map = (
        {f'k{i}': f'b{i}' for i in sub_cut_sites}
        | {f'b{i}': f'k{i}' for i in sub_cut_sites}
    )
    
    sub_cut_rho_purity = (
        (sub_cut_rho & sub_cut_rho.reindex(tranpose_map))
        .contract()
    )
    
    cut_score, cut_state, cut_overlap = get_dominant_eigenvector(sub_cut_rho, False)

    cut_state_fermion_parity = compute_fermion_parity(cut_state, sub_cut_sites)

    edm = generate_edm_from_cut_state(
        cut_state,
        sub_cut_sites,
        domains_dict['num_defect_sites']
    )

    defect_ops_results = solve_for_boundary_operators(
        edm,
        num_iters=20
    )

    left_defect_sites = list(range(
        max(domains_dict['left_projector_sites']) + 1,
        max(domains_dict['left_projector_sites']) + 1 + domains_dict['num_defect_sites']
    ))
    right_defect_sites = list(range(
        min(domains_dict['right_projector_sites']) - domains_dict['num_defect_sites'],
        min(domains_dict['right_projector_sites'])
    ))
    
    left_rdm = (
        cut_rho
        .reindex({
            f'b{i}': f'k{i}'
            for i in cut_sites if i not in left_defect_sites
        })
    )
    left_rdm = left_rdm.contract()
    left_fuse_map = [
        ('k_left', [f'k{i}' for i in left_defect_sites]),
        ('b_left', [f'b{i}' for i in left_defect_sites])
    ]
    left_rdm.fuse(left_fuse_map, inplace=True)
    
    right_rdm = (
        cut_rho
        .reindex({
            f'b{i}': f'k{i}'
            for i in cut_sites if i not in right_defect_sites
        })
    )
    right_rdm = right_rdm.contract()
    right_fuse_map = [
        ('k_right', [f'k{i}' for i in right_defect_sites]),
        ('b_right', [f'b{i}' for i in right_defect_sites])
    ]
    right_rdm.fuse(right_fuse_map, inplace=True)

    left_defect_op, right_defect_op = defect_ops_results[0]

    np_left_rdm = (
        left_rdm
        .transpose('k_left', 'b_left')
        .data
    )

    np_left_defect_op = (
        left_defect_op
        .transpose('k_left', 'b_left')
        .data
        .T
    )

    fp_list = [np_I, np_Z]
    np_fp = multikron([
        fp_list[i%2]
        for i in range(domains_dict['num_defect_sites'])
    ])

    left_defect_op_invariant = np.trace(
        np_fp
        @ np_left_defect_op.conj().T
        @ np_fp
        @ np_left_defect_op
        @ np_left_rdm
    )

    np_right_rdm = (
        right_rdm
        .transpose('k_right', 'b_right')
        .data
    )
    
    np_right_defect_op = (
        right_defect_op
        .transpose('k_right', 'b_right')
        .data
        .T
    )

    right_defect_op_invariant = np.trace(
        np_fp
        @ np_right_defect_op.conj().T
        @ np_fp
        @ np_right_defect_op
        @ np_right_rdm
    )

    np_sym_defect_m = get_multisite_op(
        np_M_reindexed,
        domains_dict['num_defect_sites']
    )

    np_defect_I = multikron([
        np_I
        for i in range(domains_dict['num_defect_sites'])
    ])

    unitary_sym_op_list = [
        np_defect_I,
        np_sym_defect_m,
        np_fp,
        np_sym_defect_m.conj()
    ]

    left_cocyles = get_cocycles_from_rho(
        np_left_rdm,
        np_left_defect_op,
        unitary_sym_op_list
    )

    right_cocyles = get_cocycles_from_rho(
        np_right_rdm,
        np_right_defect_op,
        unitary_sym_op_list
    )

    all_cocyles = np.array([left_cocyles, right_cocyles])

    cocycle_equation_output = compute_cocyle_equation_from_cocycles(all_cocyles)
    bosonic_cocycle_equation_output = (
        compute_bosonic_cocyle_equation_from_cocycles(all_cocyles)
    )

    cocycle_invariants = (all_cocyles[:,0,0]**2)*(all_cocyles[:,1,1])

    left_bosonic_cocycle = get_bosonic_cocycle(
        np_left_rdm,
        np_left_defect_op,
        unitary_sym_op_list
    )
    right_bosonic_cocycle = get_bosonic_cocycle(
        np_right_rdm,
        np_right_defect_op,
        unitary_sym_op_list
    )

    left_bosonic_cocyle_eq_from_boosnic_cocycle = (
        compute_bosonic_cocyle_equation_from_bosonic_cocycle(
            left_bosonic_cocycle
        )
    )

    right_bosonic_cocyle_eq_from_boosnic_cocycle = (
        compute_bosonic_cocyle_equation_from_bosonic_cocycle(
            right_bosonic_cocycle
        )
    )

    out = {
        'left_proj_vec': left_proj_vec,
        'raw_left_proj_vec': raw_left_proj_vec,
        'left_proj_vec_results': left_proj_vec_results,
        'right_proj_vec': right_proj_vec,
        'raw_right_proj_vec': raw_right_proj_vec,
        'right_proj_vec_results': right_proj_vec_results,
        'cut_rho_trace': cut_rho_trace,
        'cut_rho_purity': cut_rho_purity,
        'sub_cut_rho_trace': sub_cut_rho_trace,
        'sub_cut_rho_purity': sub_cut_rho_purity,
        'cut_score': cut_score,
        'cut_state': cut_state,
        'cut_state_fermion_parity': cut_state_fermion_parity,
        'cut_overlap': cut_overlap,
        'defect_ops_scores': defect_ops_results[1],
        'left_defect_op': left_defect_op,
        'right_defect_op': right_defect_op,
        'left_defect_op_invariant': left_defect_op_invariant,
        'right_defect_op_invariant': right_defect_op_invariant,
        'cocycles': all_cocyles,
        'cocycle_equation_output': cocycle_equation_output,
        'bosonic_cocycle_equation_output': bosonic_cocycle_equation_output,
        'cocycle_invariants': cocycle_invariants,
        'left_bosonic_cocycle': left_bosonic_cocycle,
        'right_bosonic_cocycle': right_bosonic_cocycle,
        'left_bosonic_cocyle_eq_from_boosnic_cocycle': left_bosonic_cocyle_eq_from_boosnic_cocycle,
        'right_bosonic_cocyle_eq_from_boosnic_cocycle': right_bosonic_cocyle_eq_from_boosnic_cocycle,
    }

    return out

## Find dominant eigenvector

In [64]:
def random_uniform_complex(shape):
    return np.random.uniform(size=shape) + 1j*np.random.uniform(size=shape)

In [65]:
def lanczos_iteration(rho, v, sites):
    w = (
        rho & v.reindex({f'k{i}': f'b{i}' for i in sites})
    ).contract()

    w_norm = np.sqrt((w & w.conj()).contract())

    out = w/w_norm

    return out

In [66]:
def multiply_state_by_jw_string(state, sites):
    gates = [
        qu_Z.reindex({'k': f'k{i}', 'b': f'b{i}'})
        for i in sites if (i%2 == 1)
    ]

    out = qtn.TensorNetwork(
        [
            *gates,
            state.reindex({f'k{i}': f'b{i}' for i in sites if (i%2 == 1)})
        ]
    )

    return out.contract()

In [67]:
def lanczos_iteration_jw_even(rho, v, sites):
    w = multiply_state_by_jw_string(v, sites)
    w = (v+w)/2

    w = (
        rho & w.reindex({f'k{i}': f'b{i}' for i in sites})
    ).contract()

    w = (multiply_state_by_jw_string(w, sites) + w)/2

    w_norm = np.sqrt((w & w.conj()).contract())

    out = w/w_norm

    return out

In [68]:
def lanczos_algorithm(rho, v, sites, num_iters=20, jw_even=False):
    update_func = lanczos_iteration_jw_even if jw_even else lanczos_iteration
    for _ in range(num_iters):
        v = update_func(rho, v, sites)

    return v

In [69]:
def get_dominant_eigenvector(rho, jw_even=False):
    k_inds = [i for i in rho.inds if i.startswith('k')]
    #b_inds = [i for i in rho.inds if i.startswith('b')]

    sites = [int(s[1:]) for s in k_inds]

    v = qtn.Tensor(
        data=random_uniform_complex((2,)*len(sites)),
        inds=[f'k{i}' for i in sites]
    )

    v = lanczos_algorithm(
        rho,
        v,
        sites,
        num_iters=20,
        jw_even=jw_even
    )

    update_func = lanczos_iteration_jw_even if jw_even else lanczos_iteration
    new_v = update_func(rho, v, sites)

    overlap = np.abs((v & new_v.conj()).contract())

    score = (
        v.reindex({f'k{i}': f'b{i}' for i in sites})
        & rho
        & v.conj()
    )
    score = score.contract()


    return score, v, overlap

In [70]:
def compute_fermion_parity(state, sites):
    odd_sites = [i for i in sites if (i%2 == 1)]

    gates = [
        qu_Z.reindex({'k': f'k{i}', 'b': f'b{i}'})
        for i in odd_sites
    ]

    gate_exp = (
        state.reindex({f'k{i}': f'b{i}' for i in odd_sites})
        & gates
        & state.conj()
    ).contract()

    return gate_exp

## Extract cocyles

In [71]:
def get_multisite_op(op, num_defect_sites):
    assert (num_defect_sites%2 == 0)

    np_op = multikron([op,]*(num_defect_sites//2))

    out = qtn.Tensor(
        np_op,
        inds=['k', 'b']
    )

    out = out.transpose('k', 'b').data

    return out

In [72]:
def get_t_squared_defect_op_from_t_defect_op(defect_op, unitary_sym_op):
    out = (
        unitary_sym_op
        @ (defect_op.conj())
        @ (unitary_sym_op.conj().T)
        @ defect_op
    )

    return out

In [73]:
def get_t_inverse_defect_op_from_t_defect_op(defect_op, unitary_sym_op):
    out = (
        (unitary_sym_op.T)
        @ (defect_op.T)
        @ (unitary_sym_op.conj())
    )

    return out

In [74]:
def get_all_defect_ops_from_t_defect_op(defect_op, unitary_sym_op):
    out = [
        np.identity(defect_op.shape[0]),
        defect_op,
        get_t_squared_defect_op_from_t_defect_op(defect_op, unitary_sym_op),
        get_t_inverse_defect_op_from_t_defect_op(defect_op, unitary_sym_op)
    ]

    return out

In [75]:
def get_cocycle(defect_rho, defect_op_list, unitary_sym_op_list, g_index,
                  h_index):
    g_op = defect_op_list[g_index]

    h_op = defect_op_list[h_index]
    h_op = h_op.conj() if (g_index%2 == 1) else h_op

    gh_index = (g_index+h_index)%4
    gh_op = defect_op_list[gh_index]

    unitary_sym_op = unitary_sym_op_list[g_index]

    out = np.trace(
        (gh_op.conj().T)
        @ unitary_sym_op
        @ h_op
        @ (unitary_sym_op.conj().T)
        @ g_op
        @ defect_rho
    )

    return out

In [76]:
def get_cocycles_from_rho(defect_rho, defect_op, unitary_sym_op_list):
    defect_op_list = get_all_defect_ops_from_t_defect_op(
        defect_op,
        unitary_sym_op_list[1]
    )

    out = [
        [
            get_cocycle(defect_rho, defect_op_list, unitary_sym_op_list, i, j)
            for j in range(1,4)
        ]
        for i in range(1,4)
    ]

    return out

In [77]:
def compute_cocyle_equation_from_cocycles(cocyles):
    shape = cocyles
    # A bit janky, oh well
    X = np.ones((2, 4, 4), dtype=complex)

    X[:, 1:, 1:] = cocyles

    out = np.zeros((2, 4, 4, 4), dtype=complex)

    for i,j,k in product(range(4), repeat=3):
        num = X[:,i,j]*X[:,(i+j)%4, k]
        if (i%2 == 0):
            denom = X[:,j,k]*X[:, i, (j+k)%4]
        if (i%2 == 1):
            denom = (X[:,j,k].conj())*X[:,i,(j+k)%4]

        #print(num)
        #print(denom)

        out[:,i,j,k] = num/denom

    return out

In [78]:
def compute_bosonic_cocyle_equation_from_cocycles(cocyles):
    bosonic_cocycles = cocyles[..., :1, :1]
    shape = bosonic_cocycles
    # A bit janky, oh well
    X = np.ones((2, 2, 2), dtype=complex)

    X[:, 1:, 1:] = bosonic_cocycles

    out = np.zeros((2, 2, 2, 2), dtype=complex)

    for i,j,k in product(range(2), repeat=3):
        num = X[:,i,j]*X[:,(i+j)%2, k]
        if (i == 0):
            denom = X[:,j,k]*X[:, i, (j+k)%2]
        if (i == 1):
            denom = (X[:,j,k].conj())*X[:,i,(j+k)%2]

        out[:,i,j,k] = num/denom

    return out

In [277]:
def get_bosonic_cocycle(defect_rho, defect_op, unitary_sym_op_list):
    defect_op_list = get_all_defect_ops_from_t_defect_op(
        defect_op,
        unitary_sym_op_list[1]
    )

    defect_op = defect_op_list[1]
    fp_defect_op = defect_op_list[2]

    unitary_sym_op = unitary_sym_op_list[1]
    fp_sym_op = unitary_sym_op_list[2]

    out = np.trace(
        (fp_defect_op.conj().T)
        @ unitary_sym_op
        @ defect_op.conj()
        @ (unitary_sym_op.conj().T)
        @ defect_op
        @ defect_rho
    )

    return out

In [279]:
def compute_bosonic_cocyle_equation_from_bosonic_cocycle(bosonic_cocycle):
    X = np.ones((2, 2), dtype=complex)

    X[1, 1] = bosonic_cocycle

    out = np.zeros((2, 2, 2), dtype=complex)

    for i,j,k in product(range(2), repeat=3):
        num = X[i,j]*X[(i+j)%2, k]
        if (i == 0):
            denom = X[j,k]*X[i, (j+k)%2]
        if (i == 1):
            denom = (X[j,k].conj())*X[i,(j+k)%2]

        out[i,j,k] = num/denom

    return out

# Check random unitaries and symmetry

In [79]:
domains_dict = {
    'num_system_sites': 24,
    'left_projector_sites': list(range(4, 8)),
    'right_projector_sites': list(range(16, 20)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 2,
    'fdlu_depth': 2,
    'fdlu_offset': 1
}

In [80]:
cluster_psi = get_cluster_state_qu_tensor_network(domains_dict['num_system_sites'])

In [81]:
rand_psi = apply_haar_random_fdlu_to_quimb_state(cluster_psi, domains_dict)

In [82]:
(rand_psi & rand_psi.conj()).contract()

np.complex128(0.999999999999996+9.71445146547012e-17j)

In [83]:
(rand_psi & cluster_psi.conj()).contract()

np.complex128(-2.423301438844973e-05+8.470329472543003e-22j)

In [84]:
symmetry_gates = [
    qu_M.reindex({
        'ks': f'k{i}', 'bs':f'b{i}',
        'kf': f'k{i+1}', 'bf':f'b{i+1}',
    })
    for i in range(0, domains_dict['num_system_sites'], 2)
]

In [85]:
sym_cluster_psi = qtn.TensorNetwork(
    [
        rand_psi.reindex({f'k{i}': f'b{i}' for i in range(domains_dict['num_system_sites'])}),
        *symmetry_gates
    ]
)

In [86]:
(
    sym_cluster_psi & sym_cluster_psi.conj()
).contract()

np.complex128(0.999999999999996+6.245004513516506e-17j)

In [87]:
(
    sym_cluster_psi & rand_psi.conj()
).contract()

np.complex128(-0.001694519117685249+5.421010862427522e-19j)

In [88]:
(
    sym_cluster_psi & rand_psi
).contract()

np.complex128(0.9999999999999962-2.7755575615628914e-17j)

In [89]:
product_psi = get_product_qu_tensor_network(domains_dict['num_system_sites'])

In [90]:
rand_psi = apply_haar_random_fdlu_to_quimb_state(product_psi, domains_dict)

In [91]:
(rand_psi & rand_psi.conj()).contract()

np.complex128(0.9999999999999976+6.938893903907228e-17j)

In [92]:
(rand_psi & product_psi.conj()).contract()

np.complex128(0.007316718750316146-4.043934725317686e-19j)

In [93]:
symmetry_gates = [
    qu_M.reindex({
        'ks': f'k{i}', 'bs':f'b{i}',
        'kf': f'k{i+1}', 'bf':f'b{i+1}',
    })
    for i in range(0, domains_dict['num_system_sites'], 2)
]

In [94]:
sym_product_psi = qtn.TensorNetwork(
    [
        rand_psi.reindex({f'k{i}': f'b{i}' for i in range(domains_dict['num_system_sites'])}),
        *symmetry_gates
    ]
)

In [95]:
(
    sym_product_psi & sym_product_psi.conj()
).contract()

np.complex128(0.9999999999999976+6.938893903907228e-17j)

In [96]:
(
    sym_product_psi & rand_psi.conj()
).contract()

np.complex128(7.0501544834351285e-06-6.776263578034403e-21j)

In [97]:
(
    sym_product_psi & rand_psi
).contract()

np.complex128(0.9999999999999976+6.938893903907228e-17j)

So random unitaries are symmetric, nice.

# Test - Cluster state

In [250]:
domains_dict = {
    'num_system_sites': 24,
    'left_projector_sites': list(range(4, 8)),
    'right_projector_sites': list(range(16, 20)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 2,
    'fdlu_depth': 0,
    'fdlu_offset': 0
}

In [251]:
cluster_psi = get_cluster_state_qu_tensor_network(domains_dict['num_system_sites'])

In [280]:
results = list()

for _ in tqdm(range(20)):
    current = find_invariants_via_projectors_from_random_state(
        cluster_psi,
        domains_dict,
        jw_even=True
    )
    results.append(current)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:12<00:00,  1.61it/s]


### Analyze results

#### Projector scores

In [281]:
np.round(np.array([
    d['left_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([ 0.,  0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0.,
       -0.,  0.,  0.,  0.,  0.,  0.,  0.])

In [282]:
np.round(np.array([
    d['right_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([ 0.,  0., -0.,  0., -0., -0.,  0.,  0.,  0.,  0., -0.,  0., -0.,
        0., -0., -0., -0.,  0., -0.,  0.])

In [283]:
def get_schmidt_vals_ratio(schmidt_vals):
    if len(schmidt_vals.data) > 1:
        return schmidt_vals.data[1]/schmidt_vals.data[0]
    else:
        return 0

In [284]:
np.round(np.array([
    get_schmidt_vals_ratio(d['left_proj_vec_results'][1])
    for d in results
]), 3)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0.])

In [285]:
np.round(np.array([
    get_schmidt_vals_ratio(d['right_proj_vec_results'][1])
    for d in results
]), 3)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0.])

#### Purities

In [286]:
np.round(np.array(
    [d['cut_rho_purity'] for d in results]
), 3)

array([1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j])

In [287]:
np.round(np.array(
    [d['sub_cut_rho_trace'] for d in results]
), 3)

array([1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j])

In [288]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j])

Purities are the same, which one could likely prove.

In [289]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j])

#### Cut state results

In [290]:
np.array(
    [d['cut_state_fermion_parity'] for d in results]
)

array([-1.-1.65894641e-17j,  1.+1.57614923e-17j, -1.+1.68364458e-18j,
        1.+2.32455906e-18j, -1.+6.40442160e-18j, -1.+1.46605758e-17j,
        1.-1.54984144e-17j,  1.-1.62873204e-17j, -1.-7.74488656e-18j,
        1.+1.12648510e-17j, -1.+7.83922033e-18j, -1.-3.44778217e-18j,
        1.-1.79214476e-17j,  1.-6.26825174e-18j, -1.+1.86046085e-17j,
        1.+5.53214637e-20j,  1.+8.28660559e-18j,  1.-1.82811774e-17j,
       -1.-1.10793690e-17j,  1.-6.81382403e-18j])

So the FP of the cut state can be even or odd. Interesting!

In [291]:
np.round(np.array(
    [d['cut_overlap'] for d in results]
), 3)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

#### Defect op scores overlaps

In [292]:
np.round(np.array([d['defect_ops_scores'][-1] for d in results]), 5)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

In [293]:
np.array([d['defect_ops_scores'][-1] for d in results])

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

In [294]:
[d['defect_ops_scores'][-1] for d in results]

[np.float64(1.0000000000000004),
 np.float64(1.0),
 np.float64(1.0000000000000004),
 np.float64(1.0),
 np.float64(1.0000000000000004),
 np.float64(1.0000000000000004),
 np.float64(1.0000000000000004),
 np.float64(1.0),
 np.float64(1.0000000000000004),
 np.float64(1.0000000000000004),
 np.float64(1.0),
 np.float64(1.0000000000000004),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0000000000000004),
 np.float64(0.9999999999999999),
 np.float64(1.0),
 np.float64(0.9999999999999999),
 np.float64(1.0),
 np.float64(0.9999999999999996)]

#### Phases

In [295]:
left_phases = np.array([
    d['left_defect_op_invariant'] for d in results
])

In [296]:
left_phases.shape

(20,)

In [297]:
np.round(left_phases, 3)

array([-1.+0.j, -1.-0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.+0.j, -1.-0.j,
       -1.-0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.-0.j, -1.-0.j,
       -1.+0.j, -1.-0.j, -1.-0.j, -1.-0.j, -1.+0.j, -1.+0.j])

In [298]:
right_phases = np.array([
    d['right_defect_op_invariant'] for d in results
])

In [299]:
right_phases.shape

(20,)

In [300]:
np.round(right_phases, 3)

array([-1.+0.j, -1.+0.j, -1.-0.j, -1.-0.j, -1.-0.j, -1.+0.j, -1.-0.j,
       -1.+0.j, -1.+0.j, -1.-0.j, -1.-0.j, -1.-0.j, -1.+0.j, -1.+0.j,
       -1.+0.j, -1.-0.j, -1.-0.j, -1.-0.j, -1.+0.j, -1.+0.j])

#### Cocyle information

In [301]:
np.array([
    d['cocycle_invariants'] for d in results
])

array([[-0.99999997+2.77555172e-16j, -1.        +2.91033652e-16j],
       [-1.        +3.57290325e-22j, -1.        +2.79855339e-17j],
       [-1.        +5.55110777e-17j, -1.        -3.42423889e-17j],
       [-1.        +1.66533297e-16j, -1.        -3.71519660e-17j],
       [-1.        -1.66533430e-16j, -1.        -1.74320561e-17j],
       [-1.        +1.66533570e-16j, -1.        +2.04484296e-16j],
       [-1.        -1.66533454e-16j, -1.        -2.70104724e-16j],
       [-1.        -2.77555595e-17j, -1.        +4.39468953e-18j],
       [-1.        +1.11022303e-16j, -1.        +2.44892555e-16j],
       [-1.        +5.37494052e-22j, -1.        -1.14073091e-16j],
       [-1.        +1.94288610e-16j, -1.        -2.02174327e-16j],
       [-1.        -5.55110940e-17j, -1.        -1.20270166e-16j],
       [-0.99999998-1.66534064e-16j, -1.        +2.27353363e-16j],
       [-0.99999992-3.33066919e-16j, -1.        +5.13472616e-17j],
       [-1.        +1.66533454e-16j, -1.        +2.76243567e-1

In [302]:
cocycle_equation_array = np.array([d['cocycle_equation_output'] for d in results])
np.linalg.norm(cocycle_equation_array-1)

np.float64(6.9384468410494e-07)

In [303]:
bosonic_cocycle_equation_array = np.array([d['bosonic_cocycle_equation_output'] for d in results])
np.linalg.norm(bosonic_cocycle_equation_array-1)

np.float64(7.930066095318125e-16)

In [304]:
results[1]['cocycles'][1]

array([[ 1.-9.32851131e-18j, -1.+3.70840869e-17j,  1.-9.32851131e-18j],
       [-1.+9.32851131e-18j, -1.+9.32851131e-18j,  1.-9.32851131e-18j],
       [ 1.-9.32851131e-18j,  1.-9.32851131e-18j, -1.+3.70840869e-17j]])

In [305]:
np.array([
    d['right_bosonic_cocycle'] for d in results
])

array([1.-9.70112173e-17j, 1.-9.32851131e-18j, 1.+1.14141296e-17j,
       1.+1.23839887e-17j, 1.+5.81068540e-18j, 1.-6.81614321e-17j,
       1.+9.00349080e-17j, 1.-1.46489651e-18j, 1.-8.16308518e-17j,
       1.+3.80243637e-17j, 1.+6.73914424e-17j, 1.+5.85937725e-17j,
       1.-7.57844545e-17j, 1.-7.86389531e-18j, 1.-9.20811892e-18j,
       1.+8.72368819e-17j, 1.+8.84927599e-17j, 1.+1.31115818e-17j,
       1.-3.71008306e-17j, 1.-7.70243587e-17j])

In [306]:
np.array([
    d['left_bosonic_cocycle'] for d in results
])

array([1.-1.11021995e-16j, 1.-3.57290326e-22j, 1.+7.16127852e-23j,
       1.-5.55109947e-17j, 1.+5.55111418e-17j, 1.-5.55112364e-17j,
       1.+5.55111514e-17j, 1.+1.38777717e-17j, 1.-2.77555759e-17j,
       1.-2.66808322e-22j, 1.-6.93887205e-17j, 1.-2.01398516e-23j,
       1.+5.55116151e-17j, 1.+1.11022326e-16j, 1.-5.55111512e-17j,
       1.+1.11022295e-16j, 1.+1.11022303e-16j, 1.+1.11022282e-16j,
       1.+2.77555763e-17j, 1.-5.55111199e-17j])

In [308]:
np.linalg.norm(np.array([
    d['left_bosonic_cocyle_eq_from_boosnic_cocycle'] for d in results
]) - 1)

np.float64(5.991603270974912e-16)

In [309]:
np.linalg.norm(np.array([
    d['right_bosonic_cocyle_eq_from_boosnic_cocycle'] for d in results
]) - 1)

np.float64(5.194866554528307e-16)

In [313]:
np.round((
    results[0]['right_defect_op']
    .transpose('k_right', 'b_right')
    .data
), 3)

array([[ 0.   -0.j   ,  0.389-0.921j,  0.   +0.j   ,  0.   +0.j   ],
       [ 0.85 -0.527j,  0.   -0.j   , -0.   -0.j   , -0.   +0.j   ],
       [ 0.   +0.j   , -0.   -0.j   , -0.   +0.j   ,  0.85 -0.527j],
       [ 0.   +0.j   ,  0.   +0.j   ,  0.315+0.949j,  0.   +0.j   ]])

# Test - Cluster state - 2 site projectors and defect operators

In [186]:
domains_dict = {
    'num_system_sites': 24,
    'left_projector_sites': list(range(6, 8)),
    'right_projector_sites': list(range(16, 18)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 2,
    'fdlu_depth': 0,
    'fdlu_offset': 0
}

In [187]:
cluster_psi = get_cluster_state_qu_tensor_network(domains_dict['num_system_sites'])

In [188]:
results = list()

for _ in tqdm(range(20)):
    current = find_invariants_via_projectors_from_random_state(
        cluster_psi,
        domains_dict,
        jw_even=True
    )
    results.append(current)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:13<00:00,  1.51it/s]


### Analyze results

#### Projector scores

In [189]:
np.round(np.array([
    d['left_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0.])

In [190]:
np.round(np.array([
    d['right_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0.])

In [191]:
def get_schmidt_vals_ratio(schmidt_vals):
    if len(schmidt_vals.data) > 1:
        return schmidt_vals.data[1]/schmidt_vals.data[0]
    else:
        return 0

In [192]:
np.round(np.array([
    get_schmidt_vals_ratio(d['left_proj_vec_results'][1])
    for d in results
]), 3)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0.])

In [193]:
np.round(np.array([
    get_schmidt_vals_ratio(d['right_proj_vec_results'][1])
    for d in results
]), 3)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0.])

#### Purities

In [194]:
np.round(np.array(
    [d['cut_rho_purity'] for d in results]
), 3)

array([1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j])

In [195]:
np.round(np.array(
    [d['sub_cut_rho_trace'] for d in results]
), 3)

array([1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j])

In [196]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j])

Purities are the same, which one could likely prove.

In [197]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j])

#### Cut state results

In [198]:
np.array(
    [d['cut_state_fermion_parity'] for d in results]
)

array([ 1.-9.00032539e-18j,  1.-6.85681180e-18j, -1.+1.37557564e-17j,
       -1.+3.06422573e-18j, -1.+1.36303484e-17j, -1.+9.40913820e-18j,
        1.-1.04879063e-17j,  1.-1.90152056e-17j, -1.-1.22526300e-17j,
        1.-1.82753166e-17j, -1.-1.87063670e-17j, -1.-7.97158050e-18j,
       -1.+1.92357622e-17j,  1.-3.39973510e-18j,  1.-9.96417547e-18j,
       -1.-1.29282890e-17j, -1.+2.80771326e-18j, -1.-1.52901341e-17j,
       -1.+1.64643322e-17j,  1.+1.84430835e-17j])

So the FP of the cut state can be even or odd. Interesting!

In [199]:
np.round(np.array(
    [d['cut_overlap'] for d in results]
), 3)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

#### Defect op scores overlaps

In [200]:
np.round(np.array([d['defect_ops_scores'][-1] for d in results]), 5)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

In [201]:
np.array([d['defect_ops_scores'][-1] for d in results])

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

In [202]:
[d['defect_ops_scores'][-1] for d in results]

[np.float64(1.0000000000000002),
 np.float64(0.9999999999999999),
 np.float64(0.9999999999999998),
 np.float64(0.9999999999999998),
 np.float64(1.0),
 np.float64(1.0000000000000002),
 np.float64(1.0000000000000002),
 np.float64(1.0000000000000002),
 np.float64(1.0000000000000004),
 np.float64(1.0),
 np.float64(1.0000000000000004),
 np.float64(0.9999999999999999),
 np.float64(0.9999999999999999),
 np.float64(1.0),
 np.float64(1.0000000000000002),
 np.float64(1.0),
 np.float64(1.0000000000000004),
 np.float64(0.9999999999999998),
 np.float64(1.0),
 np.float64(1.0)]

#### Phases

In [203]:
left_phases = np.array([
    d['left_defect_op_invariant'] for d in results
])

In [204]:
left_phases.shape

(20,)

In [205]:
np.round(left_phases, 3)

array([-1.+0.j, -1.-0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.+0.j,
       -1.-0.j, -1.-0.j, -1.+0.j, -1.-0.j, -1.+0.j, -1.-0.j, -1.+0.j,
       -1.-0.j, -1.+0.j, -1.-0.j, -1.+0.j, -1.+0.j, -1.-0.j])

In [206]:
right_phases = np.array([
    d['right_defect_op_invariant'] for d in results
])

In [207]:
right_phases.shape

(20,)

In [208]:
np.round(right_phases, 3)

array([-1.+0.j, -1.-0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.+0.j, -1.+0.j,
       -1.-0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.+0.j, -1.+0.j, -1.+0.j,
       -1.-0.j, -1.-0.j, -1.-0.j, -1.-0.j, -1.-0.j, -1.-0.j])

#### Cocyle information

In [209]:
np.array([
    d['cocycle_invariants'] for d in results
])

array([[-1.        +3.88578059e-16j, -0.99999997+3.33066902e-16j],
       [-1.        -3.33066907e-16j, -1.        -1.66533454e-16j],
       [-0.99999999+6.33174062e-17j, -0.99999996-1.23692553e-24j],
       [-1.        +1.66533454e-16j, -1.        +1.66533454e-16j],
       [-0.99999999+4.13590295e-25j, -1.        -8.32667268e-17j],
       [-0.99999999+6.93889422e-18j, -0.99999992-2.77555721e-17j],
       [-0.99999992+2.22044590e-16j, -1.        +1.32270621e-27j],
       [-1.        -1.66533454e-16j, -1.        -1.66533453e-16j],
       [-0.99999996+2.88121068e-24j, -1.        -3.15282574e-32j],
       [-0.99999997+3.33066901e-16j, -1.        +3.27353216e-25j],
       [-1.        -5.55111512e-17j, -1.        +5.55111512e-17j],
       [-1.        +3.88578059e-16j, -1.        +5.55111512e-17j],
       [-1.        -1.66533454e-16j, -1.        +1.11022302e-16j],
       [-1.        +1.66533454e-16j, -1.        +1.11022302e-16j],
       [-1.        -1.66533454e-16j, -1.        -2.22044605e-1

In [210]:
cocycle_equation_array = np.array([d['cocycle_equation_output'] for d in results])
np.linalg.norm(cocycle_equation_array-1)

np.float64(8.244881450803614e-07)

In [211]:
results[0]['bosonic_cocycle_equation_output']

array([[[[1.+0.00000000e+00j, 1.+0.00000000e+00j],
         [1.+0.00000000e+00j, 1.+0.00000000e+00j]],

        [[1.+0.00000000e+00j, 1.+0.00000000e+00j],
         [1.+0.00000000e+00j, 1.-2.77555756e-16j]]],


       [[[1.+0.00000000e+00j, 1.+0.00000000e+00j],
         [1.+0.00000000e+00j, 1.+0.00000000e+00j]],

        [[1.+0.00000000e+00j, 1.+0.00000000e+00j],
         [1.+0.00000000e+00j, 1.-2.22044608e-16j]]]])

In [216]:
np.array([
    d['cocycles'][1,1,1]
    for d in results
])

array([-0.99999997+1.11022301e-16j, -1.        -5.55111512e-17j,
       -0.99999996+0.00000000e+00j, -1.        +5.55111512e-17j,
       -1.        -2.77555756e-17j, -0.99999992-2.46519033e-32j,
       -1.        +0.00000000e+00j, -1.        -5.55111513e-17j,
       -1.        +0.00000000e+00j, -1.        +0.00000000e+00j,
       -1.        +0.00000000e+00j, -1.        +0.00000000e+00j,
       -1.        +0.00000000e+00j, -1.        +2.77555756e-17j,
       -1.        -5.55111512e-17j, -1.        -5.55111511e-17j,
       -0.99999998-1.06612476e-32j, -0.99999999+0.00000000e+00j,
       -1.        +0.00000000e+00j, -0.99999999-5.55111512e-17j])

In [212]:
bosonic_cocycle_equation_array = np.array([d['bosonic_cocycle_equation_output'] for d in results])
np.linalg.norm(bosonic_cocycle_equation_array-1)

np.float64(7.311422159359832e-16)

# Test - Product state - 2 site projectors and defect operators

In [217]:
domains_dict = {
    'num_system_sites': 24,
    'left_projector_sites': list(range(6, 8)),
    'right_projector_sites': list(range(16, 18)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 2,
    'fdlu_depth': 0,
    'fdlu_offset': 0
}

In [218]:
product_psi = get_product_qu_tensor_network(domains_dict['num_system_sites'])

In [219]:
results = list()

for _ in tqdm(range(20)):
    current = find_invariants_via_projectors_from_random_state(
        product_psi,
        domains_dict,
        jw_even=True
    )
    results.append(current)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:08<00:00,  2.25it/s]


### Analyze results

#### Projector scores

In [220]:
np.round(np.array([
    d['left_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([ 0., -0.,  0., -0.,  0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,
        0., -0.,  0.,  0., -0.,  0.,  0.])

In [221]:
np.round(np.array([
    d['right_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([-0.,  0.,  0., -0.,  0., -0.,  0., -0., -0.,  0., -0., -0., -0.,
       -0., -0.,  0.,  0., -0., -0., -0.])

In [222]:
def get_schmidt_vals_ratio(schmidt_vals):
    if len(schmidt_vals.data) > 1:
        return schmidt_vals.data[1]/schmidt_vals.data[0]
    else:
        return 0

In [223]:
np.round(np.array([
    get_schmidt_vals_ratio(d['left_proj_vec_results'][1])
    for d in results
]), 3)

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [224]:
np.round(np.array([
    get_schmidt_vals_ratio(d['right_proj_vec_results'][1])
    for d in results
]), 3)

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

#### Purities

In [225]:
np.round(np.array(
    [d['cut_rho_purity'] for d in results]
), 3)

array([1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j])

In [226]:
np.round(np.array(
    [d['sub_cut_rho_trace'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j])

In [227]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j])

Purities are the same, which one could likely prove.

In [228]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j])

#### Cut state results

In [229]:
np.array(
    [d['cut_state_fermion_parity'] for d in results]
)

array([1.+8.90010554e-19j, 1.+1.69358134e-18j, 1.+1.55680695e-18j,
       1.-8.63299678e-19j, 1.-1.63825843e-18j, 1.-1.52556906e-18j,
       1.-1.52695360e-18j, 1.+1.60521111e-18j, 1.-5.40123819e-20j,
       1.-8.49521104e-19j, 1.-1.37855154e-18j, 1.-7.91472344e-19j,
       1.-6.43685295e-19j, 1.+1.24728586e-18j, 1.+1.28531014e-18j,
       1.+1.29265271e-18j, 1.-8.27134753e-19j, 1.+1.04500583e-18j,
       1.-9.73205913e-19j, 1.+6.64301129e-19j])

So the FP of the cut state can be even or odd. Interesting!

In [230]:
np.round(np.array(
    [d['cut_overlap'] for d in results]
), 3)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

#### Defect op scores overlaps

In [231]:
np.round(np.array([d['defect_ops_scores'][-1] for d in results]), 5)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

In [232]:
np.array([d['defect_ops_scores'][-1] for d in results])

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

In [233]:
[d['defect_ops_scores'][-1] for d in results]

[np.float64(1.0000000000000002),
 np.float64(1.0000000000000004),
 np.float64(1.0),
 np.float64(0.9999999999999999),
 np.float64(1.0),
 np.float64(1.0000000000000002),
 np.float64(1.0),
 np.float64(1.0000000000000002),
 np.float64(0.9999999999999999),
 np.float64(1.0),
 np.float64(1.0000000000000002),
 np.float64(0.9999999999999999),
 np.float64(0.9999999999999999),
 np.float64(1.0000000000000002),
 np.float64(1.0000000000000004),
 np.float64(1.0),
 np.float64(1.0000000000000007),
 np.float64(1.0000000000000004),
 np.float64(1.0000000000000002),
 np.float64(0.9999999999999998)]

#### Phases

In [234]:
left_phases = np.array([
    d['left_defect_op_invariant'] for d in results
])

In [235]:
left_phases.shape

(20,)

In [236]:
np.round(left_phases, 3)

array([1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j])

In [237]:
right_phases = np.array([
    d['right_defect_op_invariant'] for d in results
])

In [238]:
right_phases.shape

(20,)

In [239]:
np.round(right_phases, 3)

array([1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j])

#### Cocyle information

In [240]:
np.array([
    d['cocycle_invariants'] for d in results
])

array([[1.+2.48739139e-16j, 1.+2.48739139e-16j],
       [1.-2.19797988e-16j, 1.-1.64286837e-16j],
       [1.+1.68549909e-16j, 1.+1.26916546e-16j],
       [1.-1.57469137e-17j, 1.+9.52753887e-17j],
       [1.+1.84574358e-16j, 1.+2.67841085e-16j],
       [1.-3.14377162e-16j, 1.-3.21316056e-16j],
       [1.+6.48131702e-18j, 1.+6.19924682e-17j],
       [1.+2.04562025e-17j, 1.+7.59673537e-17j],
       [1.-2.39144427e-16j, 1.-2.53022215e-16j],
       [1.+2.99148875e-16j, 1.+1.91812860e-16j],
       [1.+1.18154009e-16j, 1.+9.03984335e-17j],
       [1.+1.56530535e-16j, 1.+1.04488831e-16j],
       [1.+3.23893930e-16j, 1.+3.23677089e-16j],
       [1.-4.50351796e-16j, 1.-5.42292141e-16j],
       [1.+5.66053716e-17j, 1.+1.39872098e-16j],
       [1.+2.32475174e-16j, 1.+6.59417199e-17j],
       [1.-4.57183868e-16j, 1.-3.87794929e-16j],
       [1.-7.47087592e-17j, 1.-5.82288861e-17j],
       [1.-5.93436964e-16j, 1.-5.51803601e-16j],
       [1.-4.25888745e-16j, 1.-4.12010957e-16j]])

In [241]:
cocycle_equation_array = np.array([d['cocycle_equation_output'] for d in results])
np.linalg.norm(cocycle_equation_array-1)

np.float64(2.709042765715046e-14)

In [242]:
results[0]['bosonic_cocycle_equation_output']

array([[[[1.+0.00000000e+00j, 1.+0.00000000e+00j],
         [1.+0.00000000e+00j, 1.+0.00000000e+00j]],

        [[1.+0.00000000e+00j, 1.+0.00000000e+00j],
         [1.+0.00000000e+00j, 1.+1.75077951e-16j]]],


       [[[1.+0.00000000e+00j, 1.+0.00000000e+00j],
         [1.+0.00000000e+00j, 1.+0.00000000e+00j]],

        [[1.+0.00000000e+00j, 1.+0.00000000e+00j],
         [1.+0.00000000e+00j, 1.+1.75077951e-16j]]]])

In [243]:
np.array([
    d['cocycles'][1,1,1]
    for d in results
])

array([1.+7.36611877e-17j, 1.-1.01021572e-16j, 1.+1.12850937e-16j,
       1.+3.17584629e-17j, 1.+6.15247862e-17j, 1.-1.20983140e-16j,
       1.+2.16043901e-18j, 1.+1.14446634e-17j, 1.-7.62453621e-17j,
       1.+4.42774205e-17j, 1.+2.55068819e-17j, 1.+8.57148324e-17j,
       1.+1.05651679e-16j, 1.-2.17771481e-16j, 1.+3.64740107e-19j,
       1.+7.74917245e-17j, 1.-2.03279845e-16j, 1.+1.75978054e-17j,
       1.-1.88560463e-16j, 1.-1.28085127e-16j])

In [244]:
bosonic_cocycle_equation_array = np.array([d['bosonic_cocycle_equation_output'] for d in results])
np.linalg.norm(bosonic_cocycle_equation_array-1)

np.float64(1.1285984068408978e-15)

# Old code

## Debug - plug in expected defect operators

In [170]:
np.round(results[0]['left_proj_vec'].data, 3)

array([[ 0.63+0.777j,  0.  +0.j   ],
       [-0.  -0.j   ,  0.  +0.j   ]])

In [171]:
np.round(results[0]['right_proj_vec'].data, 3)

array([[0.502+0.865j, 0.   +0.j   ],
       [0.   +0.j   , 0.   +0.j   ]])

In [172]:
cut_state = results[0]['cut_state']

In [153]:
results[0]['left_defect_op']

Tensor(shape=(4, 4), inds=('b_left', 'k_left'), tags=oset([]))

In [154]:
results[0]['right_defect_op']

Tensor(shape=(4, 4), inds=('b_right', 'k_right'), tags=oset([]))

In [155]:
left_defect_op = qtn.Tensor(
    np.kron(np_X, np_X),
    inds=['k_left', 'b_left']
)

right_defect_op = qtn.Tensor(
    np.kron(np_I, np_X),
    inds=['k_right', 'b_right']
)

In [156]:
sub_cut_sites = list(range(
    max(domains_dict['left_projector_sites'])+1,
    min(domains_dict['right_projector_sites'])
))

In [157]:
rand_psi = cluster_psi

In [158]:
cut_state_fermion_parity = compute_fermion_parity(cut_state, sub_cut_sites)

edm = generate_edm_from_cut_state(
    cut_state,
    sub_cut_sites,
    domains_dict['num_defect_sites']
)

defect_ops_results = solve_for_boundary_operators(
    edm,
    num_iters=20
)

left_defect_sites = list(range(
    max(domains_dict['left_projector_sites']) + 1,
    max(domains_dict['left_projector_sites']) + 1 + domains_dict['num_defect_sites']
))
right_defect_sites = list(range(
    min(domains_dict['right_projector_sites']) - domains_dict['num_defect_sites'],
    min(domains_dict['right_projector_sites'])
))

left_rdm = (
    cut_state
    & cut_state.conj().reindex({
        f'k{i}': f'b{i}'
        for i in left_defect_sites
    })
)
left_rdm = left_rdm.contract()
left_fuse_map = [
    ('k_left', [f'k{i}' for i in left_defect_sites]),
    ('b_left', [f'b{i}' for i in left_defect_sites])
]
left_rdm.fuse(left_fuse_map, inplace=True)

right_rdm = (
    cut_state
    & cut_state.conj().reindex({
        f'k{i}': f'b{i}'
        for i in right_defect_sites
    })
)
right_rdm = right_rdm.contract()
right_fuse_map = [
    ('k_right', [f'k{i}' for i in right_defect_sites]),
    ('b_right', [f'b{i}' for i in right_defect_sites])
]
right_rdm.fuse(right_fuse_map, inplace=True)

#left_defect_op, right_defect_op = defect_ops_results[0]

np_left_rdm = (
    left_rdm
    .transpose('k_left', 'b_left')
    .data
)

np_left_defect_op = (
    left_defect_op
    .transpose('k_left', 'b_left')
    .data
    .T
)

fp_list = [np_I, np_Z]
np_fp = multikron([
    fp_list[i%2]
    for i in range(domains_dict['num_defect_sites'])
])

left_defect_op_invariant = np.trace(
    np_fp
    @ np_left_defect_op.conj().T
    @ np_fp
    @ np_left_defect_op
    @ np_left_rdm
)

np_right_rdm = (
    right_rdm
    .transpose('k_right', 'b_right')
    .data
)

np_right_defect_op = (
    right_defect_op
    .transpose('k_right', 'b_right')
    .data
    .T
)

right_defect_op_invariant = np.trace(
    np_fp
    @ np_right_defect_op.conj().T
    @ np_fp
    @ np_right_defect_op
    @ np_right_rdm
)

np_sym_defect_m = get_multisite_op(
    np_M_reindexed,
    domains_dict['num_defect_sites']
)

np_defect_I = multikron([
    np_I
    for i in range(domains_dict['num_defect_sites'])
])

unitary_sym_op_list = [
    np_defect_I,
    np_sym_defect_m,
    np_fp,
    np_sym_defect_m.conj()
]

left_cocyles = get_cocycles_from_rho(
    np_left_rdm,
    np_left_defect_op,
    unitary_sym_op_list
)

right_cocyles = get_cocycles_from_rho(
    np_right_rdm,
    np_right_defect_op,
    unitary_sym_op_list
)

all_cocyles = np.array([left_cocyles, right_cocyles])

cocycle_equation_output = compute_cocyle_equation_from_cocycles(all_cocyles)

cocycle_invariants = (all_cocyles[:,0,0]**2)*(all_cocyles[:,1,1])

In [159]:
all_cocyles[1].shape

(3, 3)

In [160]:
np.round(all_cocyles[1])

array([[ 1.+0.j, -1.+0.j,  1.+0.j],
       [-1.+0.j, -1.+0.j,  1.+0.j],
       [ 1.+0.j,  1.+0.j, -1.+0.j]])

In [161]:
defect_rho = np_right_defect_op

In [165]:
simple_defects_op_list = [
    np.kron(np_I, np_I),
    np.kron(np_I, np_X),
    np.kron(np_I, np_I),
    np.kron(np_I, np_X)
]

In [175]:
defect_rho = np_right_rdm

In [177]:
np.round(defect_rho, 3)

array([[0.5+0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0. -0.j, 0. +0.j, 0. +0.j, 0. -0.j],
       [0. -0.j, 0. +0.j, 0. +0.j, 0. -0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0.5+0.j]])

In [178]:
simple_cocycles = np.array([
    [
        get_cocycle(defect_rho, simple_defects_op_list, unitary_sym_op_list, i, j).item()
        for j in range(1,4)
    ]
    for i in range(1,4)
])

In [179]:
simple_cocycles

array([[ 0.+1.j,  1.+0.j,  0.+1.j],
       [-1.+0.j,  1.+0.j, -1.+0.j],
       [ 0.-1.j,  1.+0.j,  0.-1.j]])

In [180]:
simple_cocycle_equation = compute_cocyle_equation_from_cocycles(simple_cocycles)

In [181]:
np.linalg.norm(simple_cocycle_equation-1)

np.float64(1.0877919644084146e-15)

In [182]:
(simple_cocycles[0,0]**2)*(simple_cocycles[1,1])

np.complex128(-1.0000000000000007+0j)

In [183]:
bosonic_cocycles = simple_cocycles[:1, :1]
shape = bosonic_cocycles
# A bit janky, oh well
X = np.ones((2, 2), dtype=complex)

X[1:, 1:] = bosonic_cocycles

out = np.zeros((2, 2, 2), dtype=complex)

for i,j,k in product(range(2), repeat=3):
    num = X[i,j]*X[(i+j)%2, k]
    if (i == 0):
        denom = X[j,k]*X[i, (j+k)%2]
    if (i == 1):
        denom = (X[j,k].conj())*X[i,(j+k)%2]

    out[i,j,k] = num/denom

In [184]:
out

array([[[ 1.+0.j,  1.+0.j],
        [ 1.+0.j,  1.+0.j]],

       [[ 1.+0.j,  1.+0.j],
        [ 1.+0.j, -1.+0.j]]])

In [185]:
np.linalg.norm(out - 1)

np.float64(2.0)

In [ ]:
bosonic_cocycle_equation_output = (
    compute_bosonic_cocyle_equation_from_cocycles(all_cocyles)
)`